<a href="https://colab.research.google.com/github/MRMRAC/2025-NN/blob/main/HW_7_IGOR1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

This project requires Python 3.10 or above:

In [ ]:
import sys

assert sys.version_info >= (3, 10)

Are we using Colab or Kaggle?

In [ ]:
IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "kaggle_secrets" in sys.modules

If using Colab, the TorchMetrics library is not pre-installed so we must install it manually:

In [ ]:
if IS_COLAB:
    %pip install -qU torchmetrics

if IS_KAGGLE:
    %pip install -qU transformers

We also need PyTorch ≥ 2.6.0:

In [ ]:
from packaging.version import Version
import torch

assert Version(torch.__version__) >= Version("2.6.0")

This chapter can be very slow without a hardware accelerator, so if we can find one, let's use it:

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cuda'

Let's issue a warning if there's no hardware accelerator available:

In [ ]:
if device == "cpu":
    print("Neural nets can be very slow without a hardware accelerator.")
    if IS_COLAB:
        print("Go to Runtime > Change runtime and select a GPU hardware "
              "accelerator.")
    if IS_KAGGLE:
        print("Go to Settings > Accelerator and select GPU.")

As we did in earlier chapters, let's define the default font sizes to make the figures prettier:

In [ ]:
import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

Let's use the same `evaluate_tm()` and `train()` functions as in the previous chapters:

In [ ]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs, patience=2, factor=0.5, epoch_callback=None):
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=patience, factor=factor)
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        if epoch_callback is not None:
            epoch_callback(model, epoch)
        for index, (X_batch, y_batch) in enumerate(train_loader):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
            train_metric = metric.compute().item()
            print(f"\rBatch {index + 1}/{len(train_loader)}", end="")
            print(f", loss={total_loss/(index+1):.4f}", end="")
            print(f", {train_metric=:.2%}", end="")
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric)
        val_metric = evaluate_tm(model, valid_loader, metric).item()
        history["valid_metrics"].append(val_metric)
        scheduler.step(val_metric)
        print(f"\rEpoch {epoch + 1}/{n_epochs},                      "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.2%}, "
              f"valid metric: {history['valid_metrics'][-1]:.2%}")
    return history



As we will build and download pretty big models, we will need to free the GPU RAM regularly to avoid running out of space. For this, we will delete the models and tensors as we go, using the `del_vars()` function below. It deletes the given variables (if they exist) then calls Python's garbage collector, and it also calls `torch.cuda.empty_cache()` if we're using a CUDA GPU:

In [ ]:
import gc

def del_vars(variable_names=[]):
    for name in variable_names:
        try:
            del globals()[name]
        except KeyError:
            pass  # ignore variables that have already been deleted
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

**WARNING**: When running a Jupyter/Colab notebook, the output of each cell gets saved in the `Out` dictionary, so if the output of a cell is a large model or tensor, then it's not enough to delete the variable, you must also delete the output from the `Out` dictionary (e.g., by clearing the whole dictionary with `Out.clear()`).

## Prepare the NMT dataset
This section downloads and prepares the NMT dataset exactly like in the previous chapter.

In [ ]:
import torch.nn as nn
from datasets import load_dataset
from torch.utils.data import random_split, DataLoader
import tokenizers

In [ ]:
from datasets import load_dataset

def convert(example):
    return {
        "source_text": example["translation"]["en"],
        "target_text": example["translation"]["ru"]
    }

# загрузка датасета
nmt_original_valid_set, nmt_test_set = load_dataset(
    "opus_books", "en-ru",
    split=["train[:90%]", "train[90%:]"]
)

# преобразование структуры
nmt_original_valid_set = nmt_original_valid_set.map(convert)
nmt_test_set = nmt_test_set.map(convert)

# разделение train / validation
split = nmt_original_valid_set.train_test_split(train_size=0.8, seed=42)

nmt_train_set = split["train"]
nmt_valid_set = split["test"]

In [ ]:
def train_en_ru():
    for pair in nmt_train_set:
        yield pair["source_text"]
        yield pair["target_text"]

max_length = 500
vocab_size = 10_000
nmt_tokenizer_model = tokenizers.models.BPE(unk_token="<unk>")
nmt_tokenizer = tokenizers.Tokenizer(nmt_tokenizer_model)
nmt_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
nmt_tokenizer.enable_truncation(max_length=max_length)
nmt_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
nmt_tokenizer_trainer = tokenizers.trainers.BpeTrainer(
    vocab_size=vocab_size, special_tokens=["<pad>", "<unk>", "<s>", "</s>"])
nmt_tokenizer.train_from_iterator(train_en_ru(), nmt_tokenizer_trainer)

In [ ]:
from collections import namedtuple

fields = ["src_token_ids", "src_mask", "tgt_token_ids", "tgt_mask"]
class NmtPair(namedtuple("NmtPairBase", fields)):
    def to(self, device):
        return NmtPair(self.src_token_ids.to(device), self.src_mask.to(device),
                       self.tgt_token_ids.to(device), self.tgt_mask.to(device))

In [ ]:
def nmt_collate_fn(batch):
    src_texts = [pair['source_text'] for pair in batch]
    tgt_texts = [f"<s> {pair['target_text']} </s>" for pair in batch]
    src_encodings = nmt_tokenizer.encode_batch(src_texts)
    tgt_encodings = nmt_tokenizer.encode_batch(tgt_texts)
    src_token_ids = torch.tensor([enc.ids for enc in src_encodings])
    tgt_token_ids = torch.tensor([enc.ids for enc in tgt_encodings])
    src_mask = torch.tensor([enc.attention_mask for enc in src_encodings])
    tgt_mask = torch.tensor([enc.attention_mask for enc in tgt_encodings])
    inputs = NmtPair(src_token_ids, src_mask,
                     tgt_token_ids[:, :-1], tgt_mask[:, :-1])
    labels = tgt_token_ids[:, 1:]
    return inputs, labels

batch_size = 64
nmt_train_loader = DataLoader(nmt_train_set, batch_size=batch_size,
                              collate_fn=nmt_collate_fn, shuffle=True)
nmt_valid_loader = DataLoader(nmt_valid_set, batch_size=batch_size,
                              collate_fn=nmt_collate_fn)
nmt_test_loader = DataLoader(nmt_test_set, batch_size=batch_size,
                             collate_fn=nmt_collate_fn)

# Attention Is All You Need: The Original Transformer Architecture

## Positional encodings

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PositionalEmbedding(nn.Module):
    def __init__(self, max_length, embed_dim, dropout=0.1):
        super().__init__()
        self.pos_embed = nn.Parameter(torch.randn(max_length, embed_dim) * 0.02)
        self.dropout = nn.Dropout(dropout)

    def forward(self, X):
        return self.dropout(X + self.pos_embed[:X.size(1)])

In [ ]:
embed_dim = 512
pos_embedding = PositionalEmbedding(max_length, embed_dim)
embeddings = torch.randn(256, 500, 512)
embeddings_with_pos = pos_embedding(embeddings)
embeddings_with_pos.shape

torch.Size([256, 500, 512])

## Multi-Head Attention (MHA)

Let's implement a custom multi-head attention module to better understand how it works:

In [ ]:
class MultiheadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.h = num_heads
        self.d = embed_dim // num_heads
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, X):
        return X.view(X.size(0), X.size(1), self.h, self.d).transpose(1, 2)

    def forward(self, query, key, value, attn_mask=None, key_padding_mask=None):
        q = self.split_heads(self.q_proj(query))  # (B, h, Lq, d)
        k = self.split_heads(self.k_proj(key))  # (B, h, Lk, d)
        v = self.split_heads(self.v_proj(value))  # (B, h, Lv, d) with Lv=Lk
        scores = q @ k.transpose(2, 3) / self.d**0.5  # (B, h, Lq, Lk)

        # Masking support:
        if attn_mask is not None:
            scores = scores.masked_fill(attn_mask, -torch.inf)  # (B, h, Lq, Lk)
        if key_padding_mask is not None:
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)  # (B, 1, 1, Lk)
            scores = scores.masked_fill(mask, -torch.inf)  # (B, h, Lq, Lk)

        weights = scores.softmax(dim=-1)  # (B, h, Lq, Lk)
        Z = self.dropout(weights) @ v  # (B, h, Lq, d)
        Z = Z.transpose(1, 2)  # (B, Lq, h, d)
        Z = Z.reshape(Z.size(0), Z.size(1), self.h * self.d)  # (B, Lq, h × d)
        return (self.out_proj(Z), weights)  # (B, Lq, h × d)

The shapes are indicated with the following abbreviations:
* `B`: batch size
* `h`: number of heads
* `Lq`: max query length (in the batch)
* `Lk`: max key length (in the batch)
* `Lv`: max value length (in the batch). Values must have the same length as keys, so Lv=Lk.
* `d`: embed_dim // h

Note that PyTorch's `nn.MultiheadAttention` layer groups the three input projection layer parameters into just two parameters: `in_proj_weight` (shape (_embed_dim_ * 3, _embed_dim_) and `in_proj_bias` (shape (_embed_dim_ * 3,)).

To get the same results with the `nn.MultiheadAttention` module, you must set `batch_first=True` in the constructor. Also, if you don't need the weights, you can set `need_weights=False` when calling the `forward()` method: in many cases, this allows the module to optimize the computations, giving the same attention output, but the weights will be set to `None`.

## Building the rest of the Transformer

In [ ]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiheadAttention(d_model, nhead, dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, src, src_mask=None, src_key_padding_mask=None):
        attn, _ = self.self_attn(src, src, src, attn_mask=src_mask,
                                 key_padding_mask=src_key_padding_mask)
        Z = self.norm1(src + self.dropout(attn))
        ff = self.dropout(self.linear2(self.dropout(self.linear1(Z).relu())))
        return self.norm2(Z + ff)

In [ ]:
class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiheadAttention(d_model, nhead, dropout)
        self.multihead_attn = MultiheadAttention(d_model, nhead, dropout)
        self.dropout = nn.Dropout(dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None,
                tgt_key_padding_mask=None, memory_key_padding_mask=None):
        attn1, _ = self.self_attn(tgt, tgt, tgt,
                                  attn_mask=tgt_mask,
                                  key_padding_mask=tgt_key_padding_mask)
        Z = self.norm1(tgt + self.dropout(attn1))
        attn2, _ = self.multihead_attn(Z, memory, memory, attn_mask=memory_mask,
                                       key_padding_mask=memory_key_padding_mask)
        Z = self.norm2(Z + self.dropout(attn2))
        ff = self.dropout(self.linear2(self.dropout(self.linear1(Z).relu())))
        return self.norm3(Z + ff)

The following modules are straightforward and were left as an exercise to the reader:

In [ ]:
from copy import deepcopy

class TransformerEncoder(nn.Module):
    def __init__(self, encoder_layer, num_layers, norm=None):
        super().__init__()
        self.layers = nn.ModuleList([deepcopy(encoder_layer)
                                     for _ in range(num_layers)])
        self.norm = norm

    def forward(self, src, mask=None, src_key_padding_mask=None):
        Z = src
        for layer in self.layers:
            Z = layer(Z, mask, src_key_padding_mask)
        if self.norm is not None:
            Z = self.norm(Z)
        return Z

In [ ]:
class TransformerDecoder(nn.Module):
    def __init__(self, decoder_layer, num_layers, norm=None):
        super().__init__()
        self.layers = nn.ModuleList([deepcopy(decoder_layer)
                                     for _ in range(num_layers)])
        self.norm = norm

    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None,
                tgt_key_padding_mask=None, memory_key_padding_mask=None):
        Z = tgt
        for layer in self.layers:
            Z = layer(Z, memory, tgt_mask, memory_mask,
                      tgt_key_padding_mask, memory_key_padding_mask)
        if self.norm is not None:
            Z = self.norm(Z)
        return Z

In [ ]:
class Transformer(nn.Module):
    def __init__(self, d_model=512, nhead=8, num_encoder_layers=6,
                 num_decoder_layers=6, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        encoder_layer = TransformerEncoderLayer(d_model, nhead, dim_feedforward,
                                                dropout)
        norm1 = nn.LayerNorm(d_model)
        self.encoder = TransformerEncoder(encoder_layer, num_encoder_layers,
                                          norm1)
        decoder_layer = TransformerDecoderLayer(d_model, nhead, dim_feedforward,
                                                dropout)
        norm2 = nn.LayerNorm(d_model)
        self.decoder = TransformerDecoder(decoder_layer, num_decoder_layers,
                                          norm2)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None, memory_mask=None,
                src_key_padding_mask=None, tgt_key_padding_mask=None,
                memory_key_padding_mask=None):
        memory = self.encoder(src, src_mask, src_key_padding_mask)
        output = self.decoder(tgt, memory, tgt_mask, memory_mask,
                              tgt_key_padding_mask, memory_key_padding_mask)
        return output

# Building an English-to-Russian Transformer

In [ ]:
class NmtTransformer(nn.Module):
    def __init__(self, vocab_size, max_length, embed_dim=512, pad_id=0,
                 num_heads=8, num_layers=6, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.pos_embed = PositionalEmbedding(max_length, embed_dim, dropout)
        self.transformer = nn.Transformer(
            embed_dim, num_heads, num_encoder_layers=num_layers,
            num_decoder_layers=num_layers, batch_first=True)
        self.output = nn.Linear(embed_dim, vocab_size)

    def forward(self, pair):
        src_embeds = self.pos_embed(self.embed(pair.src_token_ids))
        tgt_embeds = self.pos_embed(self.embed(pair.tgt_token_ids))
        src_pad_mask = ~pair.src_mask.bool()
        tgt_pad_mask = ~pair.tgt_mask.bool()
        size = [pair.tgt_token_ids.size(1)] * 2
        full_mask = torch.full(size, True, device=tgt_pad_mask.device)
        causal_mask = torch.triu(full_mask, diagonal=1)
        out_decoder = self.transformer(src_embeds, tgt_embeds,
                                       src_key_padding_mask=src_pad_mask,
                                       memory_key_padding_mask=src_pad_mask,
                                       tgt_mask=causal_mask, #tgt_is_causal=True,
                                       tgt_key_padding_mask=tgt_pad_mask)
        return self.output(out_decoder).permute(0, 2, 1)

In [ ]:
torch.triu(torch.full((5, 5), True), diagonal=1)

tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]])

In [ ]:
nn.Transformer.generate_square_subsequent_mask(5)

tensor([[0., -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0.]])

In [ ]:
torch.manual_seed(42)
nmt_tr_model = NmtTransformer(vocab_size, max_length, embed_dim=128, pad_id=0,
                              num_heads=4, num_layers=2, dropout=0.1).to(device)
if device == "mps":
    # WORKAROUND: on MPS devices, we use our custom Transformer because the
    # nn.Transformer module explodes during training, see PyTorch issue #141287
    nmt_tr_model.transformer = Transformer(
        embed_dim=128, num_heads=4, num_encoder_layers=2, num_decoder_layers=2)

n_epochs = 20
xentropy = nn.CrossEntropyLoss(ignore_index=0)  # ignore <pad> tokens
optimizer = torch.optim.NAdam(nmt_tr_model.parameters())
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=vocab_size)
accuracy = accuracy.to(device)

history = train(nmt_tr_model, optimizer, xentropy, accuracy,
                nmt_train_loader, nmt_valid_loader, n_epochs)

Epoch 1/20,                      train loss: 6.3101, train metric: 3.71%, valid metric: 4.19%
Epoch 2/20,                      train loss: 5.5750, train metric: 4.52%, valid metric: 4.90%
Epoch 3/20,                      train loss: 5.0759, train metric: 5.14%, valid metric: 5.37%
Epoch 4/20,                      train loss: 4.7318, train metric: 5.65%, valid metric: 5.57%
Epoch 5/20,                      train loss: 4.5071, train metric: 5.93%, valid metric: 5.82%
Epoch 6/20,                      train loss: 4.3360, train metric: 6.26%, valid metric: 6.01%
Epoch 7/20,                      train loss: 4.1948, train metric: 6.41%, valid metric: 6.18%
Epoch 8/20,                      train loss: 4.0747, train metric: 6.63%, valid metric: 6.24%
Epoch 9/20,                      train loss: 3.9728, train metric: 6.99%, valid metric: 6.23%
Epoch 10/20,                      train loss: 3.8834, train metric: 7.22%, valid metric: 6.47%
Epoch 11/20,                      train loss: 3.7981, train

In [ ]:
torch.save(nmt_tr_model.state_dict(), "my_nmt_tr_model.pt")

In [ ]:
def translate(model, src_text, max_length=20, pad_id=0, eos_id=3):
    tgt_text = ""
    token_ids = []
    for index in range(max_length):
        batch, _ = nmt_collate_fn([{"source_text": src_text,
                                    "target_text": tgt_text}])
        with torch.no_grad():
            Y_logits = model(batch.to(device))
            Y_token_ids = Y_logits.argmax(dim=1)  # find the best token IDs
            next_token_id = Y_token_ids[0, index]  # take the last token ID

        next_token = nmt_tokenizer.id_to_token(next_token_id)
        tgt_text += " " + next_token
        if next_token_id == eos_id:
            break
    return tgt_text

In [ ]:
nmt_tr_model.eval()
translate(nmt_tr_model, "I wont to play minecraft")

' Я не могу быть спокоен в две надцать тысяч . </s>'

# Чат бот

In [1]:
# --- Loading pretrained model
model_id = "sberbank-ai/rugpt3small_based_on_gpt2"  # или "microsoft/DialoGPT-small" для лучших диалогов

euroMoE_tokenizer = AutoTokenizer.from_pretrained(model_id)
# Добавляем pad_token для GPT-2
euroMoE_tokenizer.pad_token = euroMoE_tokenizer.eos_token

euroMoE = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto", dtype="auto")
euroMoE.eval()

# --- Define generate function (важно добавить!)
def generate(model, tokenizer, prompt, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)

    # Переносим на то же устройство, что и модель
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Декодируем только новые токены
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response

# --- Define Chat Bot class
class AssistantHelper:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.context = [{"role": "system",
             "content": "Ты — чатбот ассистент. "
             "Ты разговариваешь с людьми и помогаешь им в краткой форме."
             }]

    def chat(self, prompt):
        # Добавляем сообщение пользователя
        self.context.append({"role": "user", "content": prompt})

        # Для GPT-2 нужно вручную форматировать контекст
        context_text = self.format_context()

        # Генерируем ответ
        output = generate(self.model, self.tokenizer, context_text, max_new_tokens=200)

        # Добавляем ответ в контекст
        self.context.append({"role": "assistant", "content": output})

        return output

    def format_context(self):
        """Форматирует контекст для GPT-2"""
        formatted = ""
        for message in self.context:
            if message["role"] == "system":
                formatted += f"System: {message['content']}\n"
            elif message["role"] == "user":
                formatted += f"User: {message['content']}\n"
            elif message["role"] == "assistant":
                formatted += f"Assistant: {message['content']}\n"
        formatted += "Assistant:"
        return formatted

    def clear_history(self):
        """Очищает историю диалога"""
        self.context = [self.context[0]]  # оставляем только system message

# Создаем экземпляр
assist = AssistantHelper(euroMoE, euroMoE_tokenizer)

NameError: name 'AutoTokenizer' is not defined

In [ ]:
# --- Demonstration of work
print("=== Тест 1: Приветствие ===")
print(assist.chat("Привет! Как тебя зовут?"))
print("--------")

print("=== Тест 2: Вопрос ===")
print(assist.chat("Сколько будет 2+2?"))
print("--------")

print("=== Тест 3: Продолжение диалога ===")
print(assist.chat("А 5+5?"))
print("--------")

print("=== Тест 4: Контекст сохраняется ===")
print(assist.chat("Что мы обсуждали до этого?"))
print("--------")

# Проверка истории
print("=== История диалога ===")
for msg in assist.context:
    print(f"{msg['role']}: {msg['content']}")

=== Тест 1: Приветствие ===
 Javier. Javier: I am your father. Javier: THIS IS A SOULS?! Javier: I am your mother. Javier: What? Javier: I am your father. Javier: I am your mother. Javier: What? Javier: I am your father. Javier: What? Javier: I am your father. Javier: What? Javier: I am your mother. Javier: What? Javier: I am your mother. Javier: What? Javier: I am your father. Javier: What? Javier: I am your mother. Javier: What? Javier: I am your mother. Javier: What? Javier: What? Javier: I am your father. Javier:
--------
=== Тест 2: Вопрос ===
 2+2=2. Javier: None. Javier: What? Javier: I am your mother. Javier: What? Javier: I am your mother. Javier: What? Javier: What? Javier: I am your mother. Javier: What? Javier: I am your mother. Javier: What? Javier: I am your mother. Javier: What? Javier: I am your mother. Javier: What? Javier: I am your mother. Javier: What? Javier: None.@@ВТОРОЙ@@Выглядит как вызов на вакансию, на которую сам не смог найти отклика.@@ПЕРВЫЙ@@Не на эту.@@В